# 🎓 Biometric Attendance Monitoring System
### ArcFace + InsightFace | Google Colab Test Environment

---

**Hardware Target:** Raspberry Pi 5 + AI HAT + RPi Camera Module 2 + 5G HAT  
**Cloud Test:** Google Colab (GPU/CPU) with uploaded images  

**Pipeline:**
1. ⬆️ Upload face images per person
2. 🔐 Generate ArcFace embeddings (512-dim)
3. 🔍 Run face recognition against known database
4. 📋 Log attendance with timestamps
5. 📊 Visualize results

> **Note:** Picamera2 and Tkinter are RPi-specific. This notebook replaces them with file uploads + IPython/matplotlib display, keeping all recognition logic identical to the production code.

## 📦 Section 1 — Environment Setup & Dependencies

In [ ]:
# =========================================================
# CELL 1 — Install all dependencies
# Runtime: ~3-5 minutes on first run
# =========================================================
import subprocess, sys

print("[1/5] Installing OpenCV...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "opencv-python-headless"], check=True)

print("[2/5] Installing InsightFace...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "insightface==0.7.3"], check=True)

print("[3/5] Installing ONNX Runtime (GPU preferred, fallback CPU)...")
try:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnxruntime-gpu"], check=True)
    print("   ✅ onnxruntime-gpu installed")
except Exception:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnxruntime"], check=True)
    print("   ⚠️  Falling back to onnxruntime-cpu")

print("[4/5] Installing visualization dependencies...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "matplotlib", "Pillow", "tqdm", "ipywidgets", "pandas", "seaborn"], check=True)

print("[5/5] Installing gdown for sample dataset...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)

print("\n✅ All dependencies installed successfully!")

In [ ]:
# =========================================================
# CELL 2 — Core imports & global configuration
# =========================================================
import os, cv2, csv, time, pickle, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import insightface
from tqdm import tqdm
from datetime import datetime
from PIL import Image
from IPython.display import display, clear_output, HTML
from google.colab import files
import ipywidgets as widgets

warnings.filterwarnings('ignore')

# =========================================================
# GLOBAL CONFIGURATION (mirrors production hyperparameters)
# =========================================================
CONFIG = {
    # Strict Euclidean distance threshold — same as RPi production code
    "DISTANCE_THRESHOLD": 0.95,

    # InsightFace model — buffalo_l with w600k_r50 backbone
    "MODEL_NAME": "buffalo_l",

    # Detection input size (256 matches RPi AI HAT config)
    "DET_SIZE": (640, 640),   # Higher for Colab GPU accuracy testing

    # Dataset & output paths
    "DATASET_DIR": "dataset",
    "ENCODINGS_DIR": "encodings",
    "ENCODINGS_FILE": "encodings/known_faces.pkl",
    "ATTENDANCE_FILE": "attendance.csv",

    # Encoding quality filters
    "BLUR_THRESHOLD": 40,
    "MIN_IMAGES_REQUIRED": 3,   # Relaxed for Colab demo
    "INPUT_FACE_SIZE": (112, 112),

    # Use GPU if available
    "USE_GPU": True,
}

# Create directory structure
for d in [CONFIG["DATASET_DIR"], CONFIG["ENCODINGS_DIR"]]:
    os.makedirs(d, exist_ok=True)

print("Configuration loaded:")
for k, v in CONFIG.items():
    print(f"  {k:30s}: {v}")

In [ ]:
# =========================================================
# CELL 3 — Check GPU availability
# =========================================================
import subprocess

try:
    result = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                             "--format=csv,noheader"], capture_output=True, text=True)
    if result.returncode == 0:
        gpu_info = result.stdout.strip()
        print(f"✅ GPU Available: {gpu_info}")
        PROVIDERS = ['CUDAExecutionProvider', 'CPUExecutionProvider']
        CTX_ID = 0
    else:
        raise RuntimeError("No GPU")
except Exception:
    print("⚠️  No GPU found — using CPU (slower but functional)")
    print("   Tip: Runtime → Change runtime type → GPU")
    PROVIDERS = ['CPUExecutionProvider']
    CTX_ID = -1

print(f"\nONNX Providers: {PROVIDERS}")

## 📁 Section 2 — Dataset Setup

**Two options:**
- **Option A** (Recommended for testing): Auto-download sample dataset with 5 identities
- **Option B**: Upload your own face images

In [ ]:
# =========================================================
# CELL 4A — Download LFW sample dataset (5 identities)
# Uses Labeled Faces in the Wild — public benchmark dataset
# =========================================================
import urllib.request
import tarfile, shutil

SAMPLE_IDENTITIES = {
    "George_W_Bush": [
        "https://vis-www.cs.umass.edu/lfw/images/George_W_Bush/George_W_Bush_0001.jpg",
        "https://vis-www.cs.umass.edu/lfw/images/George_W_Bush/George_W_Bush_0002.jpg",
        "https://vis-www.cs.umass.edu/lfw/images/George_W_Bush/George_W_Bush_0003.jpg",
        "https://vis-www.cs.umass.edu/lfw/images/George_W_Bush/George_W_Bush_0004.jpg",
        "https://vis-www.cs.umass.edu/lfw/images/George_W_Bush/George_W_Bush_0005.jpg",
    ],
    "Colin_Powell": [
        "https://vis-www.cs.umass.edu/lfw/images/Colin_Powell/Colin_Powell_0001.jpg",
        "https://vis-www.cs.umass.edu/lfw/images/Colin_Powell/Colin_Powell_0002.jpg",
        "https://vis-www.cs.umass.edu/lfw/images/Colin_Powell/Colin_Powell_0003.jpg",
        "https://vis-www.cs.umass.edu/lfw/images/Colin_Powell/Colin_Powell_0004.jpg",
        "https://vis-www.cs.umass.edu/lfw/images/Colin_Powell/Colin_Powell_0005.jpg",
    ],
    "Tony_Blair": [
        "https://vis-www.cs.umass.edu/lfw/images/Tony_Blair/Tony_Blair_0001.jpg",
        "https://vis-www.cs.umass.edu/lfw/images/Tony_Blair/Tony_Blair_0002.jpg",
        "https://vis-www.cs.umass.edu/lfw/images/Tony_Blair/Tony_Blair_0003.jpg",
        "https://vis-www.cs.umass.edu/lfw/images/Tony_Blair/Tony_Blair_0004.jpg",
    ],
    "Ariel_Sharon": [
        "https://vis-www.cs.umass.edu/lfw/images/Ariel_Sharon/Ariel_Sharon_0001.jpg",
        "https://vis-www.cs.umass.edu/lfw/images/Ariel_Sharon/Ariel_Sharon_0002.jpg",
        "https://vis-www.cs.umass.edu/lfw/images/Ariel_Sharon/Ariel_Sharon_0003.jpg",
        "https://vis-www.cs.umass.edu/lfw/images/Ariel_Sharon/Ariel_Sharon_0004.jpg",
    ],
    "Hugo_Chavez": [
        "https://vis-www.cs.umass.edu/lfw/images/Hugo_Chavez/Hugo_Chavez_0001.jpg",
        "https://vis-www.cs.umass.edu/lfw/images/Hugo_Chavez/Hugo_Chavez_0002.jpg",
        "https://vis-www.cs.umass.edu/lfw/images/Hugo_Chavez/Hugo_Chavez_0003.jpg",
        "https://vis-www.cs.umass.edu/lfw/images/Hugo_Chavez/Hugo_Chavez_0004.jpg",
    ],
}

print("Downloading LFW sample dataset...\n")
total_downloaded = 0

for person_name, urls in SAMPLE_IDENTITIES.items():
    person_dir = os.path.join(CONFIG["DATASET_DIR"], person_name)
    os.makedirs(person_dir, exist_ok=True)
    success = 0

    for idx, url in enumerate(urls):
        try:
            save_path = os.path.join(person_dir, f"{idx}.jpg")
            urllib.request.urlretrieve(url, save_path)
            img = cv2.imread(save_path)
            if img is not None and img.size > 0:
                success += 1
        except Exception as e:
            pass  # Some URLs may be unavailable

    print(f"  ✅ {person_name}: {success}/{len(urls)} images saved")
    total_downloaded += success

print(f"\nTotal images downloaded: {total_downloaded}")
print(f"Dataset directory: {CONFIG['DATASET_DIR']}")

In [ ]:
# =========================================================
# CELL 4B — Upload YOUR OWN images (optional)
# Run this cell only if you want to add custom people
# Upload format: name_index.jpg  e.g. John_0.jpg, John_1.jpg
# =========================================================
print("Upload images named as: PersonName_0.jpg, PersonName_1.jpg ...")
print("Example: Keshav_0.jpg, Keshav_1.jpg, Anish_0.jpg ...\n")

uploaded = files.upload()

for filename, content in uploaded.items():
    # Parse person name from filename (everything before last underscore+number)
    base = os.path.splitext(filename)[0]
    parts = base.rsplit('_', 1)
    if len(parts) == 2 and parts[1].isdigit():
        person_name = parts[0]
    else:
        person_name = base

    person_dir = os.path.join(CONFIG["DATASET_DIR"], person_name)
    os.makedirs(person_dir, exist_ok=True)

    # Count existing files to avoid overwrite
    existing = len(os.listdir(person_dir))
    save_path = os.path.join(person_dir, f"{existing}.jpg")

    with open(save_path, 'wb') as f:
        f.write(content)

    print(f"  Saved: {person_name} → {save_path}")

# Show updated dataset stats
print("\nCurrent dataset:")
for person in sorted(os.listdir(CONFIG["DATASET_DIR"])):
    pdir = os.path.join(CONFIG["DATASET_DIR"], person)
    if os.path.isdir(pdir):
        n = len(os.listdir(pdir))
        print(f"  {person}: {n} images")

In [ ]:
# =========================================================
# CELL 5 — Visualize dataset samples
# =========================================================
people = [p for p in sorted(os.listdir(CONFIG["DATASET_DIR"]))
          if os.path.isdir(os.path.join(CONFIG["DATASET_DIR"], p))]

if not people:
    print("❌ No dataset found. Run Cell 4A or 4B first.")
else:
    cols = min(5, len(people))
    fig, axes = plt.subplots(len(people), cols, figsize=(cols * 3, len(people) * 3))
    fig.suptitle("Dataset Sample Images", fontsize=16, fontweight='bold', y=1.01)

    if len(people) == 1:
        axes = [axes]

    for row, person in enumerate(people):
        pdir = os.path.join(CONFIG["DATASET_DIR"], person)
        imgs = sorted(os.listdir(pdir))[:cols]

        row_axes = axes[row] if len(people) > 1 else axes[0]
        if cols == 1:
            row_axes = [row_axes]

        for col, (ax, imgf) in enumerate(zip(row_axes, imgs)):
            img = cv2.imread(os.path.join(pdir, imgf))
            if img is not None:
                img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                ax.imshow(img_rgb)
                ax.set_title(f"{person}\n#{col}", fontsize=8)
            ax.axis('off')

        # Hide unused axes in row
        for ax in row_axes[len(imgs):]:
            ax.axis('off')

    plt.tight_layout()
    plt.savefig("dataset_samples.png", dpi=100, bbox_inches='tight')
    plt.show()
    print(f"\nDataset contains {len(people)} identities")
    for p in people:
        n = len(os.listdir(os.path.join(CONFIG["DATASET_DIR"], p)))
        print(f"  {p}: {n} images")

## 🔐 Section 3 — ArcFace Encoding Generation

Generates 512-dimensional normalized embeddings per person using InsightFace's `buffalo_l` model (identical to production RPi code).

In [ ]:
# =========================================================
# CELL 6 — Load ArcFace model
# =========================================================
print("Loading ArcFace model (buffalo_l / w600k_r50)...")
print("First run will download ~500MB model weights.\n")

model_path = os.path.expanduser("~/.insightface/models/buffalo_l/w600k_r50.onnx")

# Download model if not cached
if not os.path.exists(model_path):
    print("Downloading model weights...")
    # insightface auto-downloads on first use
    _tmp = insightface.app.FaceAnalysis(name='buffalo_l', providers=PROVIDERS)
    _tmp.prepare(ctx_id=CTX_ID, det_size=(160, 160))
    del _tmp
    print("Model downloaded and cached.")

# Load raw ArcFace model for encoding generation
arcface = insightface.model_zoo.get_model(model_path, providers=PROVIDERS)
arcface.prepare(ctx_id=CTX_ID)

print(f"\n✅ ArcFace model loaded")
print(f"   Output embedding dim: 512")
print(f"   Provider: {PROVIDERS[0]}")

In [ ]:
# =========================================================
# CELL 7 — Generate embeddings for all dataset identities
# Identical logic to production encoding_generator.py
# =========================================================

def variance_of_laplacian(image):
    """Blur detection — higher is sharper."""
    return cv2.Laplacian(image, cv2.CV_64F).var()


def generate_encodings(dataset_dir, config, arcface_model):
    known_encodings = {}
    stats = {"people": 0, "images_used": 0, "images_skipped": 0}
    encoding_report = []

    people = sorted([
        p for p in os.listdir(dataset_dir)
        if os.path.isdir(os.path.join(dataset_dir, p))
    ])

    if not people:
        raise ValueError(f"No person directories found in {dataset_dir}")

    print(f"Found {len(people)} identities in dataset\n")

    for person_name in people:
        person_dir = os.path.join(dataset_dir, person_name)
        image_files = sorted([
            f for f in os.listdir(person_dir)
            if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))
        ])

        embeddings = []
        sharpness_scores = []

        print(f"Processing: {person_name} ({len(image_files)} images)")

        for image_name in tqdm(image_files, desc=f"  {person_name}", ncols=80):
            img_path = os.path.join(person_dir, image_name)
            img = cv2.imread(img_path)

            if img is None or img.size == 0:
                stats["images_skipped"] += 1
                continue

            try:
                face = cv2.resize(img, config["INPUT_FACE_SIZE"])

                gray = cv2.cvtColor(face, cv2.COLOR_BGR2GRAY)
                sharpness = variance_of_laplacian(gray)

                if sharpness < config["BLUR_THRESHOLD"]:
                    stats["images_skipped"] += 1
                    continue

                sharpness_scores.append(sharpness)
                face_rgb = cv2.cvtColor(face, cv2.COLOR_BGR2RGB)

                raw_embedding = arcface_model.get_feat(face_rgb)[0]
                norm = np.linalg.norm(raw_embedding)

                if norm == 0:
                    stats["images_skipped"] += 1
                    continue

                embedding = raw_embedding / norm
                embeddings.append(embedding)
                stats["images_used"] += 1

            except Exception as e:
                stats["images_skipped"] += 1

        if len(embeddings) < config["MIN_IMAGES_REQUIRED"]:
            print(f"  ⚠️  Skipped {person_name} — only {len(embeddings)} valid images "
                  f"(need {config['MIN_IMAGES_REQUIRED']})\n")
            continue

        # Compute mean embedding and re-normalize (L2)
        embeddings_array = np.array(embeddings)
        master_embedding = np.mean(embeddings_array, axis=0)
        master_norm = np.linalg.norm(master_embedding)

        if master_norm == 0:
            continue

        master_embedding = master_embedding / master_norm
        quality_score = round(float(np.mean(sharpness_scores)), 2)

        known_encodings[person_name] = {
            "embedding": master_embedding,
            "samples": len(embeddings),
            "quality_score": quality_score,
        }

        encoding_report.append({
            "Identity": person_name,
            "Images Used": len(embeddings),
            "Quality Score": quality_score,
            "Embedding Dim": len(master_embedding),
            "Embedding Norm": round(float(np.linalg.norm(master_embedding)), 6),
        })

        stats["people"] += 1
        print(f"  ✅ Encoded: {person_name} | samples={len(embeddings)} | quality={quality_score}\n")

    return known_encodings, stats, encoding_report


print("Starting encoding generation...\n" + "=" * 60)
t_start = time.time()

known_encodings, enc_stats, enc_report = generate_encodings(
    CONFIG["DATASET_DIR"], CONFIG, arcface
)

elapsed = time.time() - t_start

# Save encoding database
os.makedirs(CONFIG["ENCODINGS_DIR"], exist_ok=True)
with open(CONFIG["ENCODINGS_FILE"], "wb") as f:
    pickle.dump(known_encodings, f, protocol=pickle.HIGHEST_PROTOCOL)

print("=" * 60)
print("ENCODING DATABASE CREATED")
print("=" * 60)
print(f"People Encoded   : {enc_stats['people']}")
print(f"Images Used      : {enc_stats['images_used']}")
print(f"Images Skipped   : {enc_stats['images_skipped']}")
print(f"Processing Time  : {elapsed:.2f}s")
print(f"Saved to         : {CONFIG['ENCODINGS_FILE']}")

if enc_report:
    df_enc = pd.DataFrame(enc_report)
    print("\nEncoding Report:")
    display(df_enc)

In [ ]:
# =========================================================
# CELL 8 — Validate encoding database integrity
# =========================================================
print("Validating encoding database...\n")

with open(CONFIG["ENCODINGS_FILE"], "rb") as f:
    loaded_encodings = pickle.load(f)

all_valid = True
for name, data in loaded_encodings.items():
    dim = len(data["embedding"])
    norm = np.linalg.norm(data["embedding"])
    valid = (dim == 512 and abs(norm - 1.0) < 0.01)

    status = "✅" if valid else "❌"
    print(f"  {status} {name:30s} dim={dim}  norm={norm:.4f}  "
          f"samples={data['samples']}  quality={data['quality_score']}")

    if not valid:
        all_valid = False

print()
if all_valid:
    print("✅ All embeddings are valid 512-dim normalized vectors.")
    print("   Ready for recognition inference.")
else:
    print("❌ Some embeddings are invalid — regenerate encodings.")

## 🔍 Section 4 — Face Recognition & Attendance Engine

Full recognition pipeline identical to production `main.py`, adapted for image/video file input instead of Picamera2.

In [ ]:
# =========================================================
# CELL 9 — Core recognition & attendance engine
# =========================================================

class AttendanceEngine:
    """
    Production-equivalent recognition engine.
    Decoupled from RPi/Tkinter — accepts numpy frames directly.
    """

    def __init__(self, encodings_path, config, providers, ctx_id):
        self.config = config
        self.marked_today = set()
        self.attendance_log = []

        # Load face analyzer (detection + recognition)
        print("Loading InsightFace FaceAnalysis...")
        self.face_analyzer = insightface.app.FaceAnalysis(
            name=config["MODEL_NAME"],
            providers=providers
        )
        self.face_analyzer.prepare(
            ctx_id=ctx_id,
            det_size=config["DET_SIZE"]
        )
        print("  ✅ FaceAnalysis ready")

        # Load encoding database
        with open(encodings_path, "rb") as f:
            self.known_encodings = pickle.load(f)
        print(f"  ✅ Loaded {len(self.known_encodings)} known identities")

        # Validate embeddings
        for name, data in self.known_encodings.items():
            if len(data["embedding"]) != 512:
                raise ValueError(
                    f"Identity '{name}' has {len(data['embedding'])}-dim vector. "
                    f"Expected 512. Regenerate encodings."
                )

    def recognize_frame(self, frame_bgr):
        """
        Run detection + recognition on a single BGR frame.
        Returns annotated frame and list of detection results.
        """
        results = []
        annotated = frame_bgr.copy()

        faces = self.face_analyzer.get(frame_bgr)

        for face in faces:
            x1, y1, x2, y2 = face.bbox.astype(int)
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(frame_bgr.shape[1], x2), min(frame_bgr.shape[0], y2)

            embedding = face.normed_embedding
            best_match = "Unknown"
            best_distance = 10.0

            # Strict Euclidean distance match (same as production)
            for name, data in self.known_encodings.items():
                diff = embedding - data["embedding"]
                distance = float(np.linalg.norm(diff))
                if distance < best_distance:
                    best_distance = distance
                    best_match = name

            if best_distance > self.config["DISTANCE_THRESHOLD"]:
                best_match = "Unknown"

            is_known = best_match != "Unknown"
            confidence = max(0.0, (1.0 - best_distance / self.config["DISTANCE_THRESHOLD"]) * 100)
            det_score = float(face.det_score) if hasattr(face, 'det_score') else 0.0

            result = {
                "name": best_match,
                "distance": round(best_distance, 4),
                "confidence": round(confidence, 1),
                "det_score": round(det_score, 4),
                "bbox": (x1, y1, x2, y2),
                "is_known": is_known,
            }
            results.append(result)

            # Mark attendance
            if is_known and best_match not in self.marked_today:
                self.marked_today.add(best_match)
                now = datetime.now()
                entry = {
                    "Name": best_match,
                    "Date": now.strftime("%Y-%m-%d"),
                    "Time": now.strftime("%H:%M:%S"),
                    "Distance": round(best_distance, 4),
                    "Confidence_%": round(confidence, 1),
                }
                self.attendance_log.append(entry)

            # Draw bounding box
            color = (0, 255, 128) if is_known else (0, 0, 255)
            cv2.rectangle(annotated, (x1, y1), (x2, y2), color, 3)

            label = f"{best_match}"
            sub_label = f"dist:{best_distance:.2f} | conf:{confidence:.0f}%"

            # Label background
            label_y = max(y1 - 10, 20)
            (lw, lh), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_DUPLEX, 0.65, 1)
            cv2.rectangle(annotated, (x1, label_y - lh - 6), (x1 + lw + 8, label_y + 4),
                          color, -1)
            cv2.putText(annotated, label, (x1 + 4, label_y),
                        cv2.FONT_HERSHEY_DUPLEX, 0.65, (0, 0, 0), 1, cv2.LINE_AA)
            cv2.putText(annotated, sub_label, (x1, y2 + 18),
                        cv2.FONT_HERSHEY_DUPLEX, 0.45, color, 1, cv2.LINE_AA)

        # Header overlay
        header = f"Faces: {len(faces)} | Known: {sum(r['is_known'] for r in results)}"
        cv2.putText(annotated, header, (20, 38),
                    cv2.FONT_HERSHEY_DUPLEX, 0.8, (0, 255, 255), 2, cv2.LINE_AA)

        return annotated, results

    def save_attendance_csv(self, path):
        if not self.attendance_log:
            print("No attendance entries to save.")
            return
        df = pd.DataFrame(self.attendance_log)
        df.to_csv(path, index=False)
        print(f"Attendance saved to {path} ({len(df)} entries)")

    def get_attendance_df(self):
        return pd.DataFrame(self.attendance_log) if self.attendance_log else pd.DataFrame()


# Instantiate engine
engine = AttendanceEngine(
    encodings_path=CONFIG["ENCODINGS_FILE"],
    config=CONFIG,
    providers=PROVIDERS,
    ctx_id=CTX_ID,
)
print("\n✅ AttendanceEngine ready")

## 🧪 Section 5 — Testing & Validation

In [ ]:
# =========================================================
# CELL 10 — Test recognition on dataset images
# (Leave-one-out test: use last image of each person)
# =========================================================
print("Running leave-one-out recognition test...\n")

test_results = []
test_images = []
test_frames = []

people = sorted([
    p for p in os.listdir(CONFIG["DATASET_DIR"])
    if os.path.isdir(os.path.join(CONFIG["DATASET_DIR"], p))
    and p in engine.known_encodings
])

for person_name in people:
    pdir = os.path.join(CONFIG["DATASET_DIR"], person_name)
    imgs = sorted([
        f for f in os.listdir(pdir)
        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
    ])

    # Use the last image (held-out from encoding if there are enough images)
    test_img = imgs[-1]
    img_path = os.path.join(pdir, test_img)
    frame = cv2.imread(img_path)

    if frame is None:
        print(f"  ⚠️  Could not read {img_path}")
        continue

    # Resize for consistency
    frame = cv2.resize(frame, (640, 480))

    t0 = time.time()
    annotated, results = engine.recognize_frame(frame)
    latency_ms = (time.time() - t0) * 1000

    predicted = results[0]["name"] if results else "No face detected"
    distance = results[0]["distance"] if results else None
    confidence = results[0]["confidence"] if results else 0
    det_score = results[0]["det_score"] if results else 0
    correct = (predicted == person_name)

    test_results.append({
        "Ground Truth": person_name,
        "Predicted": predicted,
        "Correct": correct,
        "Distance": distance,
        "Confidence %": confidence,
        "Det Score": det_score,
        "Latency ms": round(latency_ms, 1),
        "Faces Found": len(results),
    })

    test_images.append({
        "person": person_name,
        "correct": correct,
        "annotated": annotated,
        "predicted": predicted,
    })

    status_icon = "✅" if correct else "❌"
    print(f"  {status_icon} {person_name:30s} → {predicted:30s} "
          f"dist={distance}  conf={confidence}%  {latency_ms:.0f}ms")

df_results = pd.DataFrame(test_results)
correct_count = df_results["Correct"].sum() if not df_results.empty else 0
total_count = len(df_results)
accuracy = (correct_count / total_count * 100) if total_count > 0 else 0

print(f"\n{'='*60}")
print(f"TEST RESULTS: {correct_count}/{total_count} correct ({accuracy:.1f}% accuracy)")
print(f"{'='*60}")

In [ ]:
# =========================================================
# CELL 11 — Test with unknown face (false positive check)
# Downloads a face NOT in the dataset to verify rejection
# =========================================================
import urllib.request

# Use a known person NOT in our dataset
UNKNOWN_TEST_URL = "https://vis-www.cs.umass.edu/lfw/images/Serena_Williams/Serena_Williams_0001.jpg"
unknown_path = "test_unknown.jpg"

try:
    urllib.request.urlretrieve(UNKNOWN_TEST_URL, unknown_path)
    unknown_frame = cv2.imread(unknown_path)

    if unknown_frame is not None:
        unknown_frame = cv2.resize(unknown_frame, (640, 480))
        annotated_unknown, results_unknown = engine.recognize_frame(unknown_frame)

        predicted_u = results_unknown[0]["name"] if results_unknown else "No face"
        distance_u = results_unknown[0]["distance"] if results_unknown else None

        print(f"Unknown face test:")
        print(f"  Ground Truth  : Serena_Williams (NOT in database)")
        print(f"  Predicted     : {predicted_u}")
        print(f"  Distance      : {distance_u}")
        print(f"  Threshold     : {CONFIG['DISTANCE_THRESHOLD']}")

        if predicted_u == "Unknown":
            print(f"\n✅ PASS — Unknown correctly rejected (dist={distance_u} > threshold)")
        else:
            print(f"\n⚠️  FALSE POSITIVE — Incorrectly matched to '{predicted_u}'")
            print(f"   Consider lowering DISTANCE_THRESHOLD below {CONFIG['DISTANCE_THRESHOLD']}")

        # Show result
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        axes[0].imshow(cv2.cvtColor(unknown_frame, cv2.COLOR_BGR2RGB))
        axes[0].set_title("Input: Unknown Person (Serena Williams)", fontsize=12)
        axes[0].axis('off')
        axes[1].imshow(cv2.cvtColor(annotated_unknown, cv2.COLOR_BGR2RGB))
        axes[1].set_title(f"Result: {predicted_u} (dist={distance_u})", fontsize=12)
        axes[1].axis('off')
        plt.suptitle("False Positive Rejection Test", fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.savefig("test_unknown_result.png", dpi=100, bbox_inches='tight')
        plt.show()

except Exception as e:
    print(f"Could not download unknown test image: {e}")
    print("Skipping unknown face test.")

In [ ]:
# =========================================================
# CELL 12 — Multi-face detection test
# Composites multiple people into one frame
# =========================================================
import random

people_with_images = [
    p for p in sorted(os.listdir(CONFIG["DATASET_DIR"]))
    if os.path.isdir(os.path.join(CONFIG["DATASET_DIR"], p))
    and p in engine.known_encodings
]

if len(people_with_images) >= 2:
    print(f"Building multi-face composite from {min(4, len(people_with_images))} identities...")

    selected = people_with_images[:min(4, len(people_with_images))]
    face_crops = []

    for person in selected:
        pdir = os.path.join(CONFIG["DATASET_DIR"], person)
        imgs = sorted(os.listdir(pdir))
        # Use a different image than the test one
        idx = -2 if len(imgs) >= 2 else 0
        img = cv2.imread(os.path.join(pdir, imgs[idx]))
        if img is not None:
            face_crops.append(cv2.resize(img, (200, 200)))

    if face_crops:
        # Create a wide composite image
        composite_h = 200
        composite_w = 200 * len(face_crops)
        composite = np.zeros((composite_h, composite_w, 3), dtype=np.uint8)

        for i, crop in enumerate(face_crops):
            composite[:, i*200:(i+1)*200] = crop

        print(f"Composite frame size: {composite.shape}")
        print("Running multi-face recognition...")

        t0 = time.time()
        annotated_multi, multi_results = engine.recognize_frame(composite)
        latency_multi = (time.time() - t0) * 1000

        print(f"  Faces detected: {len(multi_results)}")
        print(f"  Latency: {latency_multi:.1f}ms")
        for r in multi_results:
            print(f"  → {r['name']:30s} dist={r['distance']}  conf={r['confidence']}%")

        # Plot
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        axes[0].imshow(cv2.cvtColor(composite, cv2.COLOR_BGR2RGB))
        axes[0].set_title(f"Input: {len(face_crops)}-person composite", fontsize=12)
        axes[0].axis('off')
        axes[1].imshow(cv2.cvtColor(annotated_multi, cv2.COLOR_BGR2RGB))
        axes[1].set_title(f"Output: {len(multi_results)} detections in {latency_multi:.0f}ms", fontsize=12)
        axes[1].axis('off')
        plt.suptitle("Multi-Face Detection & Recognition Test", fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.savefig("test_multiface.png", dpi=100, bbox_inches='tight')
        plt.show()
else:
    print("⚠️  Need at least 2 identities for multi-face test.")

In [ ]:
# =========================================================
# CELL 13 — Test with uploaded image (optional)
# Upload any JPG/PNG to run live recognition
# =========================================================
print("Upload a test image to run recognition on:")
uploaded_test = files.upload()

for fname, content in uploaded_test.items():
    img_array = np.frombuffer(content, np.uint8)
    frame = cv2.imdecode(img_array, cv2.IMREAD_COLOR)

    if frame is None:
        print(f"❌ Could not decode {fname}")
        continue

    # Resize if too large
    h, w = frame.shape[:2]
    if max(h, w) > 1280:
        scale = 1280 / max(h, w)
        frame = cv2.resize(frame, (int(w * scale), int(h * scale)))

    t0 = time.time()
    annotated, results = engine.recognize_frame(frame)
    latency_ms = (time.time() - t0) * 1000

    print(f"\nFile: {fname}")
    print(f"Frame size: {frame.shape}")
    print(f"Faces detected: {len(results)}")
    print(f"Inference time: {latency_ms:.1f}ms")

    for i, r in enumerate(results):
        icon = "✅" if r["is_known"] else "👤"
        print(f"  {icon} Face {i+1}: {r['name']:30s} dist={r['distance']}  "
              f"conf={r['confidence']}%  det={r['det_score']}")

    fig, axes = plt.subplots(1, 2, figsize=(14, 7))
    axes[0].imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    axes[0].set_title("Input Image", fontsize=12)
    axes[0].axis('off')
    axes[1].imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    axes[1].set_title(f"Recognition Result ({len(results)} faces, {latency_ms:.0f}ms)", fontsize=12)
    axes[1].axis('off')
    plt.suptitle(f"Live Test: {fname}", fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f"result_{fname}", dpi=100, bbox_inches='tight')
    plt.show()

## 📊 Section 6 — Results, Analytics & Reports

In [ ]:
# =========================================================
# CELL 14 — Annotated results grid
# =========================================================
if test_images:
    n = len(test_images)
    cols = min(3, n)
    rows = (n + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols, figsize=(cols * 6, rows * 5))
    fig.suptitle("Recognition Results — Leave-One-Out Test", fontsize=16, fontweight='bold')

    if rows == 1 and cols == 1:
        axes = [[axes]]
    elif rows == 1:
        axes = [axes]

    for i, item in enumerate(test_images):
        row, col = divmod(i, cols)
        ax = axes[row][col]

        ax.imshow(cv2.cvtColor(item["annotated"], cv2.COLOR_BGR2RGB))

        title_color = "green" if item["correct"] else "red"
        status = "✓ CORRECT" if item["correct"] else "✗ INCORRECT"
        ax.set_title(f"{item['person']}\n{status}", fontsize=10, color=title_color, fontweight='bold')
        ax.axis('off')

    # Hide unused subplots
    for i in range(n, rows * cols):
        row, col = divmod(i, cols)
        axes[row][col].axis('off')

    plt.tight_layout()
    plt.savefig("recognition_results_grid.png", dpi=100, bbox_inches='tight')
    plt.show()
else:
    print("No test results yet. Run Cell 10 first.")

In [ ]:
# =========================================================
# CELL 15 — Performance analytics dashboard
# =========================================================
if not df_results.empty:
    fig = plt.figure(figsize=(18, 12))
    fig.suptitle("Attendance System — Performance Dashboard", fontsize=18, fontweight='bold', y=0.98)

    # 1. Accuracy donut
    ax1 = fig.add_subplot(2, 4, 1)
    correct = df_results["Correct"].sum()
    incorrect = len(df_results) - correct
    colors = ["#00C851", "#ff4444"]
    wedges, texts, autotexts = ax1.pie(
        [correct, incorrect],
        labels=["Correct", "Incorrect"],
        colors=colors,
        autopct="%1.0f%%",
        startangle=90,
        wedgeprops={"width": 0.5},
    )
    ax1.set_title(f"Accuracy\n{accuracy:.1f}%", fontsize=12, fontweight='bold')

    # 2. Distance distribution per person
    ax2 = fig.add_subplot(2, 4, 2)
    valid = df_results.dropna(subset=["Distance"])
    if not valid.empty:
        bar_colors = ["#00C851" if c else "#ff4444" for c in valid["Correct"]]
        bars = ax2.bar(range(len(valid)), valid["Distance"], color=bar_colors, edgecolor='white')
        ax2.axhline(y=CONFIG["DISTANCE_THRESHOLD"], color='orange', linestyle='--',
                    linewidth=2, label=f'Threshold ({CONFIG["DISTANCE_THRESHOLD"]})')
        ax2.set_xticks(range(len(valid)))
        ax2.set_xticklabels([n.split('_')[0] for n in valid["Ground Truth"]],
                            rotation=45, ha='right', fontsize=8)
        ax2.set_ylabel("Euclidean Distance")
        ax2.set_title("Distance per Identity", fontsize=12, fontweight='bold')
        ax2.legend(fontsize=8)
        ax2.set_ylim(0, max(1.2, valid["Distance"].max() * 1.2))

    # 3. Confidence distribution
    ax3 = fig.add_subplot(2, 4, 3)
    valid_conf = df_results[df_results["Confidence %"] > 0]
    if not valid_conf.empty:
        ax3.hist(valid_conf["Confidence %"], bins=10, color='#00AAFF',
                 edgecolor='white', alpha=0.85)
        ax3.axvline(valid_conf["Confidence %"].mean(), color='orange',
                    linestyle='--', linewidth=2, label=f'Mean: {valid_conf["Confidence %"].mean():.1f}%')
        ax3.set_xlabel("Confidence %")
        ax3.set_ylabel("Count")
        ax3.set_title("Confidence Distribution", fontsize=12, fontweight='bold')
        ax3.legend(fontsize=9)

    # 4. Latency per person
    ax4 = fig.add_subplot(2, 4, 4)
    ax4.bar(range(len(df_results)), df_results["Latency ms"],
            color='#AA00FF', edgecolor='white', alpha=0.85)
    ax4.axhline(y=df_results["Latency ms"].mean(), color='orange',
                linestyle='--', linewidth=2,
                label=f'Avg: {df_results["Latency ms"].mean():.0f}ms')
    ax4.set_xticks(range(len(df_results)))
    ax4.set_xticklabels([n.split('_')[0] for n in df_results["Ground Truth"]],
                         rotation=45, ha='right', fontsize=8)
    ax4.set_ylabel("Latency (ms)")
    ax4.set_title("Inference Latency", fontsize=12, fontweight='bold')
    ax4.legend(fontsize=9)

    # 5. Summary table (bottom half)
    ax5 = fig.add_subplot(2, 1, 2)
    ax5.axis('off')

    table_data = df_results[["Ground Truth", "Predicted", "Correct",
                              "Distance", "Confidence %", "Latency ms"]].copy()
    table_data["Correct"] = table_data["Correct"].map({True: "✓", False: "✗"})

    table = ax5.table(
        cellText=table_data.values,
        colLabels=table_data.columns,
        loc='center',
        cellLoc='center',
    )
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1, 1.8)

    # Color correct/incorrect rows
    for i, row_val in enumerate(df_results["Correct"]):
        color = '#e8f5e9' if row_val else '#ffebee'
        for j in range(len(table_data.columns)):
            table[i + 1, j].set_facecolor(color)

    ax5.set_title("Detailed Results Table", fontsize=12, fontweight='bold', pad=10)

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.savefig("performance_dashboard.png", dpi=120, bbox_inches='tight')
    plt.show()

    # Print summary stats
    print("\n" + "=" * 60)
    print("PERFORMANCE SUMMARY")
    print("=" * 60)
    print(f"  Accuracy         : {accuracy:.1f}% ({correct}/{total_count})")
    valid_d = df_results["Distance"].dropna()
    if not valid_d.empty:
        print(f"  Avg Distance     : {valid_d.mean():.4f}")
        print(f"  Min Distance     : {valid_d.min():.4f}")
        print(f"  Max Distance     : {valid_d.max():.4f}")
    print(f"  Avg Latency      : {df_results['Latency ms'].mean():.1f}ms")
    print(f"  Estimated FPS    : {1000 / df_results['Latency ms'].mean():.1f}")
    print(f"  Distance Thresh  : {CONFIG['DISTANCE_THRESHOLD']}")
    print("=" * 60)
else:
    print("No results to plot. Run Cell 10 first.")

In [ ]:
# =========================================================
# CELL 16 — Distance threshold sweep analysis
# Finds optimal threshold for max accuracy on this dataset
# =========================================================
if not df_results.empty and df_results["Distance"].notna().any():
    print("Running threshold sensitivity analysis...")

    thresholds = np.arange(0.3, 1.5, 0.05)
    accuracies = []
    false_positives = []
    true_positives = []

    valid_rows = df_results.dropna(subset=["Distance"])

    for thresh in thresholds:
        tp = fp = tn = fn = 0
        for _, row in valid_rows.iterrows():
            gt = row["Ground Truth"]
            dist = row["Distance"]
            predicted_at_thresh = row["Predicted"] if dist <= thresh else "Unknown"

            if predicted_at_thresh == gt:
                tp += 1
            elif predicted_at_thresh != "Unknown" and predicted_at_thresh != gt:
                fp += 1
            elif predicted_at_thresh == "Unknown" and gt != "Unknown":
                fn += 1

        total = len(valid_rows)
        acc = (tp / total * 100) if total > 0 else 0
        fp_rate = (fp / total * 100) if total > 0 else 0
        accuracies.append(acc)
        false_positives.append(fp_rate)
        true_positives.append(tp)

    optimal_idx = np.argmax(accuracies)
    optimal_thresh = thresholds[optimal_idx]
    optimal_acc = accuracies[optimal_idx]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(thresholds, accuracies, 'b-o', linewidth=2, markersize=4, label="Accuracy")
    ax1.axvline(x=CONFIG["DISTANCE_THRESHOLD"], color='orange', linestyle='--',
                linewidth=2, label=f'Current ({CONFIG["DISTANCE_THRESHOLD"]})')
    ax1.axvline(x=optimal_thresh, color='green', linestyle='--',
                linewidth=2, label=f'Optimal ({optimal_thresh:.2f}, {optimal_acc:.0f}%)')
    ax1.set_xlabel("Distance Threshold")
    ax1.set_ylabel("Accuracy %")
    ax1.set_title("Accuracy vs Threshold", fontsize=13, fontweight='bold')
    ax1.legend()
    ax1.grid(alpha=0.3)

    ax2.fill_between(thresholds, accuracies, alpha=0.3, color='blue', label='Accuracy')
    ax2.fill_between(thresholds, false_positives, alpha=0.3, color='red', label='False Positive Rate')
    ax2.plot(thresholds, accuracies, 'b-', linewidth=2)
    ax2.plot(thresholds, false_positives, 'r-', linewidth=2)
    ax2.axvline(x=optimal_thresh, color='green', linestyle='--', linewidth=2)
    ax2.set_xlabel("Distance Threshold")
    ax2.set_ylabel("Rate %")
    ax2.set_title("Accuracy / FP Tradeoff", fontsize=13, fontweight='bold')
    ax2.legend()
    ax2.grid(alpha=0.3)

    plt.suptitle("Threshold Sensitivity Analysis", fontsize=15, fontweight='bold')
    plt.tight_layout()
    plt.savefig("threshold_analysis.png", dpi=100, bbox_inches='tight')
    plt.show()

    print(f"\nCurrent threshold : {CONFIG['DISTANCE_THRESHOLD']}")
    print(f"Optimal threshold : {optimal_thresh:.2f} (achieves {optimal_acc:.0f}% accuracy)")
    if abs(optimal_thresh - CONFIG["DISTANCE_THRESHOLD"]) > 0.1:
        print(f"\n💡 Recommendation: Set DISTANCE_THRESHOLD = {optimal_thresh:.2f} in production config")
    else:
        print(f"\n✅ Current threshold is near optimal.")
else:
    print("No valid distance data. Run Cell 10 first.")

In [ ]:
# =========================================================
# CELL 17 — Attendance log & CSV export
# =========================================================
engine.save_attendance_csv(CONFIG["ATTENDANCE_FILE"])

att_df = engine.get_attendance_df()

if not att_df.empty:
    print("\nAttendance Log:")
    display(att_df)

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    axes[0].barh(att_df["Name"], att_df["Confidence_%"],
                 color='#00C851', edgecolor='white', alpha=0.85)
    axes[0].set_xlabel("Confidence %")
    axes[0].set_title("Attendance Confidence Scores", fontsize=12, fontweight='bold')
    axes[0].axvline(x=70, color='orange', linestyle='--', linewidth=1.5, label='70% line')
    axes[0].legend()

    axes[1].barh(att_df["Name"], att_df["Distance"],
                 color='#0099FF', edgecolor='white', alpha=0.85)
    axes[1].axvline(x=CONFIG["DISTANCE_THRESHOLD"], color='red',
                    linestyle='--', linewidth=1.5, label=f'Threshold ({CONFIG["DISTANCE_THRESHOLD"]})')
    axes[1].set_xlabel("Euclidean Distance")
    axes[1].set_title("Attendance Match Distances", fontsize=12, fontweight='bold')
    axes[1].legend()

    plt.suptitle("Attendance Session Report", fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig("attendance_report.png", dpi=100, bbox_inches='tight')
    plt.show()
else:
    print("No attendance entries recorded yet.")
    print("Run Cell 10 (recognition test) to populate attendance log.")

In [ ]:
# =========================================================
# CELL 18 — Inter-identity similarity heatmap
# Shows pairwise distances between all known embeddings
# =========================================================
with open(CONFIG["ENCODINGS_FILE"], "rb") as f:
    loaded_enc = pickle.load(f)

names = sorted(loaded_enc.keys())
n = len(names)

if n < 2:
    print("Need at least 2 identities for similarity matrix.")
else:
    dist_matrix = np.zeros((n, n))

    for i, name_i in enumerate(names):
        for j, name_j in enumerate(names):
            e_i = loaded_enc[name_i]["embedding"]
            e_j = loaded_enc[name_j]["embedding"]
            dist_matrix[i, j] = np.linalg.norm(e_i - e_j)

    # Short labels
    short_names = [n.replace('_', ' ').split()[-1] for n in names]

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Distance matrix
    sns.heatmap(dist_matrix, annot=True, fmt=".2f",
                xticklabels=short_names, yticklabels=short_names,
                cmap="RdYlGn_r", ax=axes[0],
                vmin=0, vmax=CONFIG["DISTANCE_THRESHOLD"] * 1.5,
                linewidths=0.5, annot_kws={"size": 9})
    axes[0].set_title("Pairwise Euclidean Distance\n(lower = more similar)",
                      fontsize=12, fontweight='bold')

    # Cosine similarity matrix (1 - cosine dist)
    cos_matrix = np.zeros((n, n))
    for i, name_i in enumerate(names):
        for j, name_j in enumerate(names):
            e_i = loaded_enc[name_i]["embedding"]
            e_j = loaded_enc[name_j]["embedding"]
            cos_sim = float(np.dot(e_i, e_j))
            cos_matrix[i, j] = cos_sim

    sns.heatmap(cos_matrix, annot=True, fmt=".3f",
                xticklabels=short_names, yticklabels=short_names,
                cmap="Blues", ax=axes[1],
                vmin=-1, vmax=1,
                linewidths=0.5, annot_kws={"size": 9})
    axes[1].set_title("Cosine Similarity Matrix\n(higher diagonal = better separation)",
                      fontsize=12, fontweight='bold')

    plt.suptitle("Embedding Space Analysis", fontsize=15, fontweight='bold')
    plt.tight_layout()
    plt.savefig("embedding_similarity.png", dpi=100, bbox_inches='tight')
    plt.show()

    # Off-diagonal min (inter-class separation)
    mask = ~np.eye(n, dtype=bool)
    off_diag = dist_matrix[mask]
    print(f"\nEmbedding Separation Analysis:")
    print(f"  Min inter-class distance  : {off_diag.min():.4f}")
    print(f"  Mean inter-class distance : {off_diag.mean():.4f}")
    print(f"  Max inter-class distance  : {off_diag.max():.4f}")
    print(f"  Threshold                 : {CONFIG['DISTANCE_THRESHOLD']}")
    separable = off_diag.min() > CONFIG["DISTANCE_THRESHOLD"]
    print(f"  Classes fully separable   : {'✅ Yes' if separable else '⚠️  No (some identities are close)'}")

## 📤 Section 7 — Export & Download Results

In [ ]:
# =========================================================
# CELL 19 — Export all results & download
# =========================================================
import zipfile

output_files = [
    CONFIG["ENCODINGS_FILE"],
    CONFIG["ATTENDANCE_FILE"],
    "dataset_samples.png",
    "recognition_results_grid.png",
    "performance_dashboard.png",
    "threshold_analysis.png",
    "attendance_report.png",
    "embedding_similarity.png",
    "test_unknown_result.png",
    "test_multiface.png",
]

zip_name = f"attendance_system_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.zip"

with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in output_files:
        if os.path.exists(f):
            zf.write(f)
            print(f"  ✅ Added: {f}")
        else:
            print(f"  ⚠️  Not found (skipped): {f}")

print(f"\nExported: {zip_name}")

# Also save results CSV
if not df_results.empty:
    csv_path = "test_results.csv"
    df_results.to_csv(csv_path, index=False)
    print(f"Test results CSV saved: {csv_path}")

print("\nDownloading results...")
files.download(zip_name)

## 🔧 Section 8 — RPi5 Production Deployment Notes

### Differences between this Colab test and RPi5 production:

| Component | Colab Test | RPi5 Production |
|-----------|------------|------------------|
| Camera | File upload / URL | Picamera2 (RPi Cam v2) |
| GPU | T4 / CPU | AI HAT (Hailo-8L NPU) |
| Display | Matplotlib | Tkinter fullscreen GUI |
| Detection size | 640×640 | 256×256 (for 30fps on Pi) |
| ONNX Provider | CUDA / CPU | `HailoExecutionProvider` or CPU |
| Attendance sync | CSV only | CSV + 5G HTTP POST |
| Threading | Single-thread | Dual-thread (capture + inference) |

### To deploy on RPi5 with 5G:

```python
# Replace PROVIDERS and CTX_ID in CONFIG with:
PROVIDERS = ['CPUExecutionProvider']  # or HailoExecutionProvider if model converted
CTX_ID = -1

# 5G attendance sync (add to _log_local_csv):
import requests
def _sync_to_server(self, payload):
    try:
        r = requests.post(
            "http://YOUR_SERVER_IP:PORT/attendance",
            json=payload,
            timeout=5
        )
        print(f"[5G SYNC] {r.status_code}")
    except Exception as e:
        print(f"[5G SYNC FAILED] {e}")
```

### Accuracy recommendations from this test:
- Collect **50+ images per person** (currently: 24s @ 0.3s interval = 80 max)
- Ensure **frontal + slight angle** coverage in dataset
- Set `DISTANCE_THRESHOLD` based on threshold sweep analysis above
- Use `det_size=(256, 256)` on RPi5 to hit 30fps target
- Enable `AwbEnable=True` in Picamera2 for consistent color